# Rainfall Data Cleaning & Annual Statistics

**Purpose:** Cleans and summarises multi-station and DWD station rainfall data
for the Ziegenrück catchment, flagging gaps and computing yearly totals.

**What it does:**
- Loads DWD gauge data and multi-station Excel files
- Replaces the −9999 missing-value sentinel with NaN
- Computes annual, daily, and per-station rainfall statistics
- Generates a monthly precipitation bar chart for a chosen year

**Input:** DWD station Excel file, `rain.xlsx` (multi-station)  
**Output:** Printed statistics, precipitation plot

---

In [ ]:
import pandas as pd
import numpy as np

# Load file
df = pd.read_excel(r"C:\Users\raah\Desktop\rain.xlsx")

# Replace missing code (-9999) with NaN
df = df.replace(-9999, np.nan)

# Ensure 'Date' column is datetime
df['Date'] = pd.to_datetime(df['Date'])

# Extract year
df['Year'] = df['Date'].dt.year

# Station columns (skip Date, Time, Year)
stations = df.columns[2:-1]  # adjust depending on your Excel structure

# Melt the dataframe to long format
long_df = df.melt(id_vars=['Date', 'Year'], value_vars=stations,
                  var_name='Station', value_name='Rainfall')

# Group by Station and Year, sum rainfall ignoring NaN
rainfall_per_year = long_df.groupby(['Station', 'Year'])['Rainfall'].sum().reset_index()

# Display the table
print("Total Rainfall per Station per Year (without missing values):")
print(rainfall_per_year)


In [ ]:
import pandas as pd
import numpy as np

# Load file
df = pd.read_excel(r"C:\Users\raah\Desktop\rain.xlsx")

# Replace missing code (-9999) with NaN
df = df.replace(-9999, np.nan)

# Ensure 'Date' column is datetime
df['Date'] = pd.to_datetime(df['Date'])

# Extract year and day (date only) from datetime
df['Year'] = df['Date'].dt.year
df['Day'] = df['Date'].dt.date  # used to group by day

# Station columns (skip Date, Time, Year)
stations = df.columns[2:]  # assuming first two columns are Date, Time

# Melt the dataframe to long format
long_df = df.melt(id_vars=['Date', 'Year', 'Day'], value_vars=stations,
                  var_name='Station', value_name='Rainfall')

# 1️⃣ Total rainfall per year per station (sum all hourly values)
rainfall_table = long_df.groupby(['Station', 'Year'])['Rainfall'].sum().unstack()

# 2️⃣ Missing days per year per station
# A day is missing if all hourly Rainfall values for that day are NaN
missing_days = (long_df.groupby(['Station', 'Day'])['Rainfall']
                          .apply(lambda x: x.isna().all())
                          .reset_index())

# Count missing days per year
missing_days['Year'] = pd.to_datetime(missing_days['Day']).dt.year
missing_table = missing_days.groupby(['Station', 'Year'])['Rainfall'].sum().unstack()

# Export both tables to Excel
output_file = r"C:\Users\raah\Desktop\rain_summary_wide.xlsx"
with pd.ExcelWriter(output_file) as writer:
    rainfall_table.to_excel(writer, sheet_name='TotalRainfall')
    missing_table.to_excel(writer, sheet_name='MissingDays')

print(f"Wide-format tables with hourly data aggregated to daily missing values exported to {output_file}")


In [ ]:
import pandas as pd
import numpy as np

# Load the Excel file and strip column names
df = pd.read_excel(r"C:\Users\raah\Downloads\tageswerte_RR_02651_19470401_20241231_hist (1)\Microsoft Excel-Arbeitsblatt (neu).xlsx")
df.columns = df.columns.str.strip()  # remove leading/trailing spaces

# Replace missing value code with NaN if any
df['RS'] = df['RS'].replace(-9999, np.nan)

# Convert MESS_DATUM to datetime
df['Date'] = pd.to_datetime(df['MESS_DATUM'], format='%Y%m%d')

# Filter dates from 01.09.1995 to 31.12.2023
start_date = '1995-09-01'
end_date = '2023-12-31'
df = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)]

# Create a continuous daily date range
full_dates = pd.date_range(start=start_date, end=end_date)
df = df.set_index('Date').reindex(full_dates).rename_axis('Date').reset_index()

# Station column (single station)
df['Station'] = 'Station_1'

# Extract year
df['Year'] = df['Date'].dt.year

# Total rainfall per year (wide format), ignoring missing values
rainfall_table = df.pivot_table(index='Station', columns='Year', values='RS', aggfunc='sum', min_count=1)

# Missing days per year (wide format)
missing_table = df.pivot_table(index='Station', columns='Year', values='RS', aggfunc=lambda x: x.isna().sum())

# Export both tables to Excel
output_file = r"C:\Users\raah\Desktop\single_station_rain_summary.xlsx"
with pd.ExcelWriter(output_file) as writer:
    rainfall_table.to_excel(writer, sheet_name='TotalRainfall')
    missing_table.to_excel(writer, sheet_name='MissingDays')

print(f"Rainfall and missing days tables exported to {output_file}")


In [ ]:
import pandas as pd
import numpy as np

# Load file
df = pd.read_excel(r"C:\Users\raah\Desktop\rain.xlsx")

# Replace missing code (-9999) with NaN
df = df.replace(-9999, np.nan)

# Ensure 'Date' column is datetime
df['Date'] = pd.to_datetime(df['Date'])

# Extract year and day (date only) from datetime
df['Year'] = df['Date'].dt.year
df['Day'] = df['Date'].dt.date  # used to group by day

# Station columns (skip Date, Time, Year)
stations = df.columns[2:]  # assuming first two columns are Date, Time

# Melt the dataframe to long format
long_df = df.melt(id_vars=['Date', 'Year', 'Day'], value_vars=stations,
                  var_name='Station', value_name='Rainfall')

# 1️⃣ Total rainfall per year per station (sum all hourly values)
rainfall_table = long_df.groupby(['Station', 'Year'])['Rainfall'].sum().unstack()

# 2️⃣ Missing days per year per station
# A day is missing if all hourly Rainfall values for that day are NaN
missing_days = (long_df.groupby(['Station', 'Day'])['Rainfall']
                          .apply(lambda x: x.isna().all())
                          .reset_index())

# Count missing days per year
missing_days['Year'] = pd.to_datetime(missing_days['Day']).dt.year
missing_table = missing_days.groupby(['Station', 'Year'])['Rainfall'].sum().unstack()

# Export both tables to Excel
output_file = r"C:\Users\raah\Desktop\rain_summary_wide.xlsx"
with pd.ExcelWriter(output_file) as writer:
    rainfall_table.to_excel(writer, sheet_name='TotalRainfall')
    missing_table.to_excel(writer, sheet_name='MissingDays')

print(f"Wide-format tables with hourly data aggregated to daily missing values exported to {output_file}")


In [ ]:
import pandas as pd
import numpy as np

# Load the CSV/Excel file
df = pd.read_csv(r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\climate_ziegenrueck\Time_series.csv", sep=None, engine='python')  # adjust path

# Only use the 'datetime' and 'precip' columns
df = df[['datetime', 'precip']]

# Replace missing value code if any (e.g., -9999)
df['precip'] = df['precip'].replace(-9999, np.nan)

# Convert 'datetime' column to datetime type
df['DateTime'] = pd.to_datetime(df['datetime'], dayfirst=True)

# Extract year and day
df['Year'] = df['DateTime'].dt.year
df['Day'] = df['DateTime'].dt.date

# Add station column
df['Station'] = 'Station_1'

# 1️⃣ Total rainfall per year (sum all hourly precip)
rainfall_table = df.groupby(['Station', 'Year'])['precip'].sum().unstack()

# 2️⃣ Missing days per year
# A day is missing if all hourly precip values are NaN
missing_days = df.groupby(['Station', 'Day'])['precip'].apply(lambda x: x.isna().all()).reset_index()
missing_days['Year'] = pd.to_datetime(missing_days['Day']).dt.year
missing_table = missing_days.groupby(['Station', 'Year'])['precip'].sum().unstack()

# Export to Excel
output_file = r"C:\Users\raah\Desktop\hourly_precip_summary.xlsx"
with pd.ExcelWriter(output_file) as writer:
    rainfall_table.to_excel(writer, sheet_name='TotalRainfall')
    missing_table.to_excel(writer, sheet_name='MissingDays')

print(f"Wide-format tables exported to {output_file}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import calendar

# ---------- USER SETTINGS ----------
input_file = r"C:\Users\raah\Desktop\rain.xlsx"
target_year = 2014   # <<-- change to any year you want
# -----------------------------------

# Load file
df = pd.read_excel(input_file)

# Parse Date (dayfirst)
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')

# Identify station columns (everything except Date and Time)
ignore_cols = {'Date', 'Time'}
station_cols = [c for c in df.columns if c not in ignore_cols]

# Replace missing-code with NaN
df[station_cols] = df[station_cols].replace(-9999, np.nan).astype(float)

# Extract Year and Month
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

# Filter for the chosen year
df_year = df[df['Year'] == target_year]
if df_year.empty:
    raise ValueError(f"No data found for year {target_year} in {input_file}.")

# Sum by month for each station (months 1..12)
monthly_totals = df_year.groupby('Month')[station_cols].sum(min_count=1)

# Ensure all months 1-12 are present (fill missing months with 0)
monthly_totals = monthly_totals.reindex(range(1, 13), fill_value=0)


# ---------- PLOT grouped bar chart ----------
plt.figure(figsize=(14, 7))
months = list(range(1, 13))
x = np.arange(len(months))

n_stations = len(station_cols)
if n_stations == 0:
    raise ValueError("No station columns detected. Check your file headers.")

bar_width = 0.8 / n_stations

for i, station in enumerate(station_cols):
    plt.bar(
        x + i * bar_width,
        monthly_totals[station].values,
        width=bar_width,
        label=station
    )

# Month labels (use short month names)
month_labels = [calendar.month_abbr[m] for m in months]
plt.xticks(x + (n_stations - 1) * bar_width / 2, month_labels)
plt.xlabel("Month")
plt.ylabel("Total Rainfall (mm)")
plt.title(f"Monthly Rainfall Totals per Station — {target_year}")
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(axis='y', alpha=0.35)
plt.tight_layout()
plt.show()

# Optionally save monthly_totals to Excel:
monthly_totals.to_excel(r"C:\Users\raah\Desktop\monthly_totals_{}.xlsx".format(target_year))


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import calendar
import csv

# ---------- USER SETTINGS ----------
rain_file = r"C:\Users\raah\Desktop\rain.xlsx"
discharge_file = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Discharge data\gesamt_ab_2009_Abfluss_stuendlich.csv"
target_year = 2017   # <<-- change to any year you want
# -----------------------------------

# ---------- LOAD RAIN DATA ----------
df_rain = pd.read_excel(rain_file)
df_rain['Date'] = pd.to_datetime(df_rain['Date'], dayfirst=True, errors='coerce')

# Identify station columns
ignore_cols = {'Date', 'Time'}
station_cols = [c for c in df_rain.columns if c not in ignore_cols]
df_rain[station_cols] = df_rain[station_cols].replace(-9999, np.nan).astype(float)

# Extract Year and Month
df_rain['Year'] = df_rain['Date'].dt.year
df_rain['Month'] = df_rain['Date'].dt.month

# Filter for the chosen year
df_rain_year = df_rain[df_rain['Year'] == target_year]
if df_rain_year.empty:
    raise ValueError(f"No rainfall data found for year {target_year}.")

# Sum by month for each station
monthly_totals_rain = df_rain_year.groupby('Month')[station_cols].sum(min_count=1)
monthly_totals_rain = monthly_totals_rain.reindex(range(1, 13), fill_value=0)

# ---------- LOAD DISCHARGE DATA ----------
# Auto-detect delimiter
with open(discharge_file, 'r', encoding='utf-8') as f:
    sample = f.read(1024)
    dialect = csv.Sniffer().sniff(sample, delimiters=[',',';','\t'])
    sep = dialect.delimiter

df_discharge = pd.read_csv(discharge_file, sep=sep)
df_discharge.columns = df_discharge.columns.str.strip()

# Detect date column
possible_date_cols = ['Zeit', 'Date', 'Datetime', 'datetime', 'Datum']
date_col = next((c for c in df_discharge.columns if c in possible_date_cols), None)
if date_col is None:
    raise ValueError("No recognizable date column found in discharge file.")

df_discharge['Zeit'] = pd.to_datetime(df_discharge[date_col], dayfirst=True, errors='coerce')
df_discharge['Year'] = df_discharge['Zeit'].dt.year
df_discharge['Month'] = df_discharge['Zeit'].dt.month

# Filter for the chosen year
df_discharge_year = df_discharge[df_discharge['Year'] == target_year]
if df_discharge_year.empty:
    raise ValueError(f"No discharge data found for year {target_year}.")

# Detect numeric discharge column automatically
numeric_cols = df_discharge.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['Year', 'Month']]
if not numeric_cols:
    raise ValueError("No numeric column found for discharge data.")
discharge_col = numeric_cols[0]

# Aggregate monthly (mean)
monthly_discharge = df_discharge_year.groupby('Month')[discharge_col].mean()
monthly_discharge = monthly_discharge.reindex(range(1, 13), fill_value=0)

# ---------- PLOT ----------
plt.figure(figsize=(14, 7))
months = list(range(1, 13))
x = np.arange(len(months))

# Rainfall bars
n_stations = len(station_cols)
bar_width = 0.8 / n_stations
bars = []

for i, station in enumerate(station_cols):
    b = plt.bar(
        x + i * bar_width,
        monthly_totals_rain[station].values,
        width=bar_width,
        label=station
    )
    bars.append(b)

# Month labels
month_labels = [calendar.month_abbr[m] for m in months]
plt.xticks(x + (n_stations - 1) * bar_width / 2, month_labels)
plt.xlabel("Month")
plt.ylabel("Total Rainfall (mm)")
plt.title(f"Monthly Rainfall and Discharge — {target_year}")
plt.grid(axis='y', alpha=0.35)

# Discharge line (secondary y-axis)
ax2 = plt.gca().twinx()
bar_center = x + (n_stations - 1) * bar_width / 2
line, = ax2.plot(bar_center, monthly_discharge.values, color='red', marker='o', linewidth=2, label='Discharge')
ax2.set_ylabel(f"Avg. Q ({discharge_col})")
ax2.grid(False)

# Legends
plt.legend(handles=[b[0] for b in bars], labels=station_cols, bbox_to_anchor=(1.05, 1), loc='upper left')
#ax2.legend(handles=[line], loc='upper right')

plt.tight_layout()
plt.show()

# ---------- SAVE RAIN + DISCHARGE IN ONE SHEET ----------
output_file = fr"C:\Users\raah\Desktop\monthly_data_{target_year}.xlsx"

# Combine rainfall and discharge into one DataFrame
combined = monthly_totals_rain.copy()
combined['Discharge'] = monthly_discharge.values  # add discharge column

# Export
combined.to_excel(output_file, sheet_name='Monthly_Data')

print(f"Combined monthly data saved to: {output_file}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import calendar
import csv

# ---------------- USER SETTINGS ----------------
rain_file = r"C:\Users\raah\Desktop\rain.xlsx"
discharge_file = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Discharge data\gesamt_ab_2009_Abfluss_stuendlich.csv"
interp_file = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Time_series_data_ZR\Loop_6_TimeSeries_ZR.csv"

target_year = 2018
# ------------------------------------------------


# =============== LOAD RAINFALL DATA ===============
df_rain = pd.read_excel(rain_file)
df_rain["Date"] = pd.to_datetime(df_rain["Date"], dayfirst=True, errors="coerce")

ignore_cols = {"Date", "Time"}
station_cols = [c for c in df_rain.columns if c not in ignore_cols]

df_rain[station_cols] = df_rain[station_cols].replace(-9999, np.nan).astype(float)

df_rain["Year"] = df_rain["Date"].dt.year
df_rain["Month"] = df_rain["Date"].dt.month

df_rain_year = df_rain[df_rain["Year"] == target_year]
if df_rain_year.empty:
    raise ValueError("No rainfall data found for this year.")

monthly_totals_rain = df_rain_year.groupby("Month")[station_cols].sum(min_count=1)
monthly_totals_rain = monthly_totals_rain.reindex(range(1, 13), fill_value=0)


# =============== LOAD INTERPOLATED PRECIP DATA ===============
# Auto-detect delimiter
with open(interp_file, "r", encoding="utf-8") as f:
    sample = f.read(1024)
    dialect = csv.Sniffer().sniff(sample, delimiters=[",", ";", "\t"])
    sep_interp = dialect.delimiter

df_interp = pd.read_csv(interp_file, sep=sep_interp)
df_interp.columns = df_interp.columns.str.strip()

# Detect datetime column
possible_date_cols = ["datetime", "Datetime", "date", "Date"]
date_col_interp = next((c for c in df_interp.columns if c in possible_date_cols), None)

if date_col_interp is None:
    raise ValueError("Interpolated file must contain a datetime column.")

df_interp["datetime"] = pd.to_datetime(df_interp[date_col_interp], dayfirst=True, errors="coerce")
df_interp["Year"] = df_interp["datetime"].dt.year
df_interp["Month"] = df_interp["datetime"].dt.month

df_interp_year = df_interp[df_interp["Year"] == target_year]
if df_interp_year.empty:
    raise ValueError("No interpolated data found for this year.")

# Detect precip column
possible_precip_cols = ["precip", "rain", "value"]
precip_col = next((c for c in df_interp.columns if c.lower() in possible_precip_cols), None)

if precip_col is None:
    raise ValueError("No precip column found in interpolated dataset.")

monthly_interp = df_interp_year.groupby("Month")[precip_col].sum(min_count=1)
monthly_interp = monthly_interp.reindex(range(1, 13), fill_value=0)

# Add to rainfall table
monthly_totals_rain["Interpolated"] = monthly_interp.values
station_cols.append("Interpolated")


# =============== LOAD DISCHARGE DATA ===============
# Auto-detect delimiter
with open(discharge_file, "r", encoding="utf-8") as f:
    sample = f.read(1024)
    dialect = csv.Sniffer().sniff(sample, delimiters=[",", ";", "\t"])
    sep = dialect.delimiter

df_discharge = pd.read_csv(discharge_file, sep=sep)
df_discharge.columns = df_discharge.columns.str.strip()

date_col_discharge = next((c for c in df_discharge.columns if c.lower() in ["zeit","date","datetime","datum"]), None)

if date_col_discharge is None:
    raise ValueError("No datetime column found in discharge data.")

df_discharge["Zeit"] = pd.to_datetime(df_discharge[date_col_discharge], dayfirst=True, errors="coerce")
df_discharge["Year"] = df_discharge["Zeit"].dt.year
df_discharge["Month"] = df_discharge["Zeit"].dt.month

df_discharge_year = df_discharge[df_discharge["Year"] == target_year]
if df_discharge_year.empty:
    raise ValueError("No discharge data found for this year.")

numeric_cols = df_discharge.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ["Year", "Month"]]
discharge_col = numeric_cols[0]

monthly_discharge = df_discharge_year.groupby("Month")[discharge_col].mean()
monthly_discharge = monthly_discharge.reindex(range(1, 13), fill_value=0)


# =============== PLOT =================
plt.figure(figsize=(14, 7))
months = range(1, 13)
x = np.arange(len(months))

n_stations = len(station_cols)
bar_width = 0.8 / n_stations
bars = []

for i, station in enumerate(station_cols):
    b = plt.bar(
        x + i * bar_width,
        monthly_totals_rain[station].values,
        width=bar_width,
        label=station
    )
    bars.append(b)

month_labels = [calendar.month_abbr[m] for m in months]
plt.xticks(x + (n_stations - 1) * bar_width / 2, month_labels)
plt.xlabel("Month")
plt.ylabel("Total Rainfall (mm)")
plt.title(f"Monthly Rainfall, Interpolated & Discharge — {target_year}")
plt.grid(axis='y', alpha=0.35)

# Discharge line
ax2 = plt.gca().twinx()
center_positions = x + (n_stations - 1) * bar_width / 2
line, = ax2.plot(center_positions, monthly_discharge.values, color='red', marker='o', linewidth=2, label='Discharge')
ax2.set_ylabel(f"Avg. Q ({discharge_col})")

# Legends
plt.legend(handles=[b[0] for b in bars], labels=station_cols, bbox_to_anchor=(1.05, 1), loc='upper left')
#ax2.legend(handles=[line], loc='upper right')

plt.tight_layout()
plt.show()



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import calendar
import csv

# ---------------- USER SETTINGS ----------------
interp_file = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Time_series_data_ZR\Loop_6_TimeSeries_ZR.csv"
discharge_file = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Discharge data\gesamt_ab_2009_Abfluss_stuendlich.csv"

target_year = 2018
# ------------------------------------------------


# =============== LOAD INTERPOLATED PRECIP DATA ===============
# Auto-detect delimiter
with open(interp_file, "r", encoding="utf-8") as f:
    sample = f.read(1024)
    dialect = csv.Sniffer().sniff(sample, delimiters=[",", ";", "\t"])
    sep_interp = dialect.delimiter

df_interp = pd.read_csv(interp_file, sep=sep_interp)
df_interp.columns = df_interp.columns.str.strip()

# Detect datetime column
possible_date_cols = ["datetime", "Datetime", "date", "Date"]
date_col_interp = next((c for c in df_interp.columns if c in possible_date_cols), None)

if date_col_interp is None:
    raise ValueError("Interpolated file must contain a datetime column.")

df_interp["datetime"] = pd.to_datetime(df_interp[date_col_interp], dayfirst=True, errors="coerce")
df_interp["Year"] = df_interp["datetime"].dt.year
df_interp["Month"] = df_interp["datetime"].dt.month

df_interp_year = df_interp[df_interp["Year"] == target_year]
if df_interp_year.empty:
    raise ValueError(f"No interpolated data found for year {target_year}.")

# Detect precip column
possible_precip_cols = ["precip", "rain", "value"]
precip_col = next((c for c in df_interp.columns if c.lower() in possible_precip_cols), None)

if precip_col is None:
    raise ValueError("No precipitation column found in interpolated dataset.")

# Monthly sum of interpolated precipitation
monthly_interp = df_interp_year.groupby("Month")[precip_col].sum(min_count=1)
monthly_interp = monthly_interp.reindex(range(1, 13), fill_value=0)


# =============== LOAD DISCHARGE DATA ===============
# Auto-detect delimiter
with open(discharge_file, "r", encoding="utf-8") as f:
    sample = f.read(1024)
    dialect = csv.Sniffer().sniff(sample, delimiters=[",", ";", "\t"])
    sep = dialect.delimiter

df_discharge = pd.read_csv(discharge_file, sep=sep)
df_discharge.columns = df_discharge.columns.str.strip()

date_col_discharge = next((c for c in df_discharge.columns if c.lower() in ["zeit","date","datetime","datum"]), None)

if date_col_discharge is None:
    raise ValueError("No datetime column found in discharge data.")

df_discharge["Zeit"] = pd.to_datetime(df_discharge[date_col_discharge], dayfirst=True, errors="coerce")
df_discharge["Year"] = df_discharge["Zeit"].dt.year
df_discharge["Month"] = df_discharge["Zeit"].dt.month

df_discharge_year = df_discharge[df_discharge["Year"] == target_year]
if df_discharge_year.empty:
    raise ValueError(f"No discharge data found for year {target_year}.")

numeric_cols = df_discharge.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ["Year", "Month"]]
discharge_col = numeric_cols[0]

# Monthly mean discharge
monthly_discharge = df_discharge_year.groupby("Month")[discharge_col].mean()
monthly_discharge = monthly_discharge.reindex(range(1, 13), fill_value=0)


# =============== PLOT (INTERP BAR + DISCHARGE LINE) ===============
plt.figure(figsize=(14, 7))
months = range(1, 13)
x = np.arange(len(months))

# Bar chart for interpolated precip
bars = plt.bar(
    x,
    monthly_interp.values,
    width=0.6,
    label="Interpolated Precip",
    color='skyblue'
)

plt.xticks(x, [calendar.month_abbr[m] for m in months])
plt.xlabel("Month")
plt.ylabel("Interpolated Precip (mm)")
plt.title(f"Interpolated Precipitation & Discharge — {target_year}")
plt.grid(axis='y', alpha=0.35)

# Discharge line on second y-axis
ax2 = plt.gca().twinx()
line, = ax2.plot(
    x,
    monthly_discharge.values,
    color='red',
    marker='o',
    linewidth=2,
    label='Discharge'
)
ax2.set_ylabel(f"Avg. Q ({discharge_col})")

# Legends
plt.legend(loc='upper left')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()





In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# -----------------------------
# Load reservoir data
# -----------------------------
reservoir_file = r"C:\Users\raah\Desktop\Study materials\Reservoir data_NT.xlsx"
df = pd.read_excel(reservoir_file)
df.columns = df.columns.str.strip()
df['Datum'] = pd.to_datetime(df['Datum'], errors='coerce', dayfirst=True)
df = df.set_index('Datum').sort_index()
df['WaterLevel'] = pd.to_numeric(df['WaterLevel'], errors='coerce')

# -----------------------------
# Load discharge data
# -----------------------------
discharge_file = r"C:\Users\raah\Desktop\Aufgabe\TALSIM_NG\EZG\Ziegenrueck\Discharge data\gesamt_ab_2009_Abfluss_stuendlich.csv"
discharge = pd.read_csv(discharge_file, sep=None, engine='python', encoding='utf-8-sig')
discharge.columns = discharge.columns.str.strip()
discharge['Zeit'] = pd.to_datetime(discharge['Zeit'], errors='coerce', dayfirst=True)
discharge = discharge.set_index('Zeit').sort_index()

# Detect discharge column
q_col = None
for c in discharge.columns:
    if c.strip().lower() in ('m3/s','m3s','abfluss','abfluss_m3s'):
        q_col = c
        break
if q_col is None:
    q_col = discharge.select_dtypes(include='number').columns[0]

# -----------------------------
# Parameters
# -----------------------------
year = 2018

# -----------------------------
# Monthly average reservoir level
# -----------------------------
df_year = df[df.index.year == year]
monthly_avg = df_year['WaterLevel'].resample('MS').mean()

# Yearly bounds for color ramp
min_level = monthly_avg.min()  # most negative → lowest water
max_level = monthly_avg.max()  # least negative → highest water

# Normalize for color ramp (0 = lowest water, 1 = highest water)
normalized = (monthly_avg - min_level) / (max_level - min_level)
colors = plt.cm.Blues(normalized)  # darker = more water

# -----------------------------
# Monthly average discharge
# -----------------------------
discharge_year = discharge[discharge.index.year == year]
monthly_discharge = discharge_year[q_col].resample('MS').mean()

# -----------------------------
# Plot
# -----------------------------
fig, ax1 = plt.subplots(figsize=(12,6))


# Bar chart for reservoir levels with dotted hatch
bars = ax1.bar(
    monthly_avg.index,
    monthly_avg.values,
    color=colors,
    width=20,
    align='center',
    edgecolor='grey',
    hatch='..'   # dotted pattern so even light bars are visible
)

ax1.set_xlabel("Month")
ax1.set_ylabel("Water Level (m)")
ax1.set_title(f"Monthly Reservoir Levels and Discharge ({year})\nDarker = More Water (vollstau = -0.595 m)")
ax1.xaxis.set_major_locator(mdates.MonthLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b'))
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# Secondary y-axis for discharge
ax2 = ax1.twinx()
ax2.plot(monthly_discharge.index, monthly_discharge.values, color='black', marker='o', linewidth=1.8, label='Monthly Avg Discharge')
ax2.set_ylabel("Discharge (m³/s)")
ax2.grid(False)

# Plot bars for reservoir levels
bars = ax1.bar(monthly_avg.index, monthly_avg.values, color=colors, width=20, align='center')

# Plot monthly average discharge on secondary y-axis
line = ax2.plot(monthly_discharge.index, monthly_discharge.values, color='black', marker='o', linewidth=1.8)[0]

# Legends using style you prefer
plt.legend(handles=[bars[0], line], 
           labels=['Reservoir Level', 'Monthly Avg Discharge'], 
           bbox_to_anchor=(1.05, 1), loc='upper left')



plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
